# Benchmark Quantum Fourier Transform

The Quantum Fourier Transform (QFT) function is a quantum algorithmic primitive, utilized algorithms such as Shor's Factoring, Discrete Logarithm  and Quantum Phase Estimation. It is the quantum analog for
discrete Fourier transform, applied on the quantum variable state vector.

The basis states $|x\rangle_n$ on $n$ qubits, where $x\in\{0,\dots, N-1\}$ and $N=2^n$, transform as

$${\cal{F}}_N|x\rangle = \frac{1}{\sqrt{N}} \sum_{x=0}^{N-1} e^{2\pi i x k/N}| k\rangle ~~.$$

Therefore, for a general state $|\psi\rangle = \sum_{x=0}^{N-1}a_x |x\rangle$,  the components of $|\chi\rangle  = {\cal{F}_N}|\psi \rangle= \sum_{k=0}^{N-1}b_k |k \rangle$.
satisfy $$b_k =\frac{1}{\sqrt{N}} \sum_{x=0}^{N-1}a_x e^{2\pi i x k/N}  $$


The transformation is achieved by Classiq's built-in function `qft`, taking an argument `target`: `QArray[QBit]` a QFT is applied to `target`.

#### The model

We evaluate the QFT of a comb distribution 

$$|\psi_0 \rangle = \frac{1}{\sqrt{2^{N-m}}}\sum_{t=0}^{2^{N-m}-1}|t 2^m\rangle~~, $$

 and compare the outcome population to the expected one


$$ |\psi_f \rangle = \frac{1}{\sqrt{2^{m}}}\sum_{t=0}^{2^m-1}|t 2^{n-m} \rangle~~.$$

We take the simple case of $m=0$, which results in the initial state

$$ |\psi_0 \rangle = \frac{1}{\sqrt{2^{n}}}\sum_{t=0}^{2^{n}-1}|t \rangle~~, $$

which in the qubit representation corresponds to 


$$|\psi_0\rangle =(|0\rangle + |1\rangle)\otimes(|0\rangle + |1\rangle)\otimes \cdots \otimes (|0\rangle + |1\rangle)  = H^{\otimes n} |0^n \rangle~~,$$

and the final state is given by 

$$|\psi_f \rangle = |0^n \rangle $$


#### The Scoring function
For a general comb state we define the score in terms of the **total variation distance** $D_{TV}(p,u) = \frac{1}{2}\sum_{i=0}^{N-1}\left|p_i-u_i \right|$, where $p_i$ and $u_i$ are the observed and expected propbabilities, correspondingly. For the comb state we have $u_{t 2^m} = 1/m$ for $t=[0,2^{m}-1]$ and zero for the rest of the probabilities. The distance is bounded between zero and $1-1/n$. To obtain a score between zero and one (one is a perfect result) we define the score as 
$$S =1 - \frac{D_{TV}(p,u)}{1-1/n} ~~.$$



## Defining the model

In [1]:
import asyncio
import datetime
import sys

sys.path.insert(0, "..")

import numpy as np
from benchmark import BenchmarkExample
from collector import ResultCollector
from hardware import HardwareRunner
from hardwares_preferences import HARDWARES

from classiq import *

In [2]:
class QFTExample(BenchmarkExample):
    def __init__(self, num_qubits: int, m: int):
        super().__init__(
            name="qft",
            num_qubits=num_qubits,
        )
        assert m < num_qubits
        self.m = m

    def create_main(self) -> callable:
        @qfunc
        def prepare_comb(m: CInt, qba: QArray):
            hadamard_transform(qba[m : qba.len])

        @qfunc
        def main(x: Output[QNum[self.num_qubits]]):
            allocate(x)
            prepare_comb(self.m, x)  # prepare the initial state, a comb state
            qft(x)

        return main

    async def submit(self, qprog: QuantumProgram) -> str:
        with ExecutionSession(qprog) as es:
            job = es.submit_sample()
            return job.id

    async def score(self, job_id):
        job = ExecutionJob.from_id(job_id)
        result = await job.result_async()
        df = result[0].value.dataframe

        n = self.num_qubits
        N = 2**n

        p = np.zeros(N)
        p[df["x"]] = df["probability"].to_numpy()

        u = np.zeros(N)
        if self.m == 0:
            u[0] = 1.0
        else:
            step = 2 ** (n - self.m)
            u[np.arange(0, N, step)] = 1.0 / (2**self.m)

        score = 1.0 - 0.5 * np.abs(p - u).sum() / (1.0 - 1.0 / n)

        exec_minutes = (job.end_time - job.start_time).total_seconds() / 60.0

        return {
            "score": float(score),
            "execution_time": exec_minutes,
        }

## Benchmarking a 4-qubits QFT

Define a `BenchmarkExample` (the models are optimized for width):

In [3]:
example = QFTExample(num_qubits=4, m=0)
example.show()

Quantum program link: https://platform.classiq.io/circuit/3B8ExnuoCOerdhQraIERghw499x


Define the number of shots and other hyperparameters:

In [4]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

Define `HardwareRunner` for each backend. Here we choose one machine for IBM, one for IonQ, as well as Classiq simulator for reference:

In [5]:
classiq_runners = [
    HardwareRunner("Classiq", backend_name, **common_config)
    for backend_name in HARDWARES["Classiq"][0:2]
]

ionq_runners = [
    HardwareRunner("IonQ", backend_name, **common_config)
    for backend_name in HARDWARES["IonQ"][0:1]
]


ibm_runners = [
    HardwareRunner(
        "IBM Quantum",
        backend_name,
        **common_config,
        backend_kwargs={
            "access_token": "PUT YOUR TOKEN HERE",
            "channel": "PUT YOUR CHANNEL HERE",
            "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
        },
    )
    for backend_name in HARDWARES["IBM Quantum"][0:1]
]

runners = [
    *classiq_runners,
    *ionq_runners,
    *ibm_runners,
]

In [6]:
print("Running for Backends:")
print(
    *[(runner.backend_service_provider, runner.backend_name) for runner in runners],
    sep="\n"
)

Running for Backends:
('Classiq', 'simulator')
('Classiq', 'simulator_statevector')
('IonQ', 'qpu.forte-1')
('IBM Quantum', 'ibm_kingston')


Define a `ResultCollector` with a given benchmark filename:

In [7]:
FILENAME = f"data/{example.name}{example.num_qubits}.csv"

In [8]:
collector = ResultCollector(FILENAME, build_each_time=True)

Run all runners:

In [9]:
print(
    "=" * 10
    + f"  {example.name}-{example.num_qubits} ({datetime.datetime.now()})   "
    + "=" * 10
)
await asyncio.gather(*[collector.run(runner, example) for runner in runners]);

==========  qft-4 (2026-03-18 21:35:46.785433)   ==========
2026-03-18 21:35:49.830143: Submit qft-4 for Classiq - simulator
2026-03-18 21:35:51.469912: Completed qft-4 for Classiq - simulator --> Score {'score': 1.0, 'execution_time': 0.011141383333333334}
** Report updated: qft-4 for Classiq - simulator


In [10]:
await collector.print_status()

========== (2026-03-18 21:35:52.258863)   ==========
qft-4 | Classiq - simulator_statevector | COMPLETED | score=1.0000 | time=0.01 min
qft-4 | IonQ - qpu.forte-1 | COMPLETED | score=0.9973 | time=15.15 min
qft-4 | IBM Quantum - ibm_kingston | COMPLETED | score=0.6853 | time=86.55 min
qft-4 | Classiq - simulator | COMPLETED | score=1.0000 | time=0.01 min
